# EEEM068 - Multi-View ViT-Small + SiT-S - Part 2

**Scope:** Preprocessing pipeline, dataset class, imbalance handling, and dataloaders

**Source notebook:** `EEEM068_SiT_S_Final_FIXED (2).ipynb`

**Notes**
- This notebook is a split component of the original final notebook.
- Some setup cells are intentionally repeated so each part is easier to understand in isolation.
- If you want to run the full pipeline end-to-end, use the original notebook or follow the README in sequence.

# EEEM068 — Knee MRI Classification · ViT-Small + SiT-S
**University of Surrey — Applied Machine Learning (Spring 2026)**

| Item | Detail |
|---|---|
| Model | Multi-View ViT-Small patch16/224 (shared backbone, 3 planes) |
| Pretrained weights | **SiT-S** — Sara Ahmed et al. 2021 (self-supervised ImageNet) |
| Dataset | MRNet — 1,370 knee MRI exams (Stanford University Medical Center) |
| Task | Multi-label: ACL tear · Meniscus tear · Abnormal |
| Loss | Focal BCE (γ=0.5) + Label smoothing + MixUp |

**Run order:** Execute cells top-to-bottom. Edit only the two paths in Section 1.

---


## Section 1 — Imports & Configuration

In [1]:
import os, io, gc, math, pickle, random, warnings, urllib.request
from pathlib import Path
from io import BytesIO
from collections import deque, OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T
import timm

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

print(f"PyTorch : {torch.__version__}")
print(f"timm    : {timm.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


PyTorch : 2.11.0+cu130
timm    : 1.0.27
CUDA    : True
GPU     : NVIDIA RTX A4000
VRAM    : 16.7 GB


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — fixed for your Surrey/otter AML folder
# ══════════════════════════════════════════════════════════════════════════════
class CFG:
    # ── MAIN PROJECT PATHS ────────────────────────────────────────────────────
    AML_DIR = Path("/user/HS402/mi00806/Downloads/AML")

    ROOT    = Path("/user/HS402/mi00806/.cache/kagglehub/datasets/"
                   "cjinny/mrnet-v1/versions/1/MRNet-v1.0")
    OUT_DIR = AML_DIR / "mrnet_outputs"

    # Your real downloaded SiT-S checkpoints are inside AML/sit_zip/
    SIT_DIR = AML_DIR / "sit_zip"
    SIT_VARIANT = "ImageNet"   # options: ImageNet, Pascal, Pets, STL10
    SIT_CACHE_PATH = str(SIT_DIR / f"SiT_Small_{SIT_VARIANT}.pth")

    CHECKPOINT_PATH = str(OUT_DIR / "best_vit.pth")
    TENSORBOARD_DIR = str(OUT_DIR / "tb_logs")

    IMAGE_SIZE  = 224
    MODEL_NAME  = "vit_small_patch16_224"
    NUM_CLASSES = 3
    LABEL_NAMES = ["acl", "meniscus", "abnormal"]

    TOP_K_SLICES = 5
    N_CHANNELS   = 3

    BATCH_SIZE  = 8
    EPOCHS      = 60
    MIN_EPOCHS  = 20
    PATIENCE    = 15
    NUM_WORKERS = 4
    SEED        = 42

    HEAD_LR           = 1e-3
    BACKBONE_LR_LATE  = 1e-5
    BACKBONE_LR_EARLY = 5e-6
    WEIGHT_DECAY      = 1e-4

    UNFREEZE_PARTIAL_EP = 5
    UNFREEZE_FULL_EP    = 10
    N_BLOCKS_PARTIAL    = 6
    WARMUP_EPOCHS       = 3
    WARMUP_RESTART      = 2

    DROP_PATH_RATE = 0.1
    DROPOUT_HEAD1  = 0.3
    DROPOUT_HEAD2  = 0.15
    LABEL_SMOOTH   = 0.05
    MIXUP_ALPHA    = 0.2
    MIXUP_PROB     = 0.4
    MIXUP_START_EP = UNFREEZE_PARTIAL_EP + 1

    FOCAL_GAMMA = 0.5
    FOCAL_ALPHA = [0.82, 0.63, 0.19]

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)

assert CFG.ROOT.exists(), f"Dataset not found: {CFG.ROOT}"
assert Path(CFG.SIT_CACHE_PATH).exists(), (
    f"SiT checkpoint not found: {CFG.SIT_CACHE_PATH}\n"
    f"Your downloaded files should be in: {CFG.SIT_DIR}\n"
    "Run: ls -lh /user/HS402/mi00806/Downloads/AML/sit_zip/"
)

print("CFG loaded ✅")
print(f"  Device     : {CFG.DEVICE}")
print(f"  Dataset    : {CFG.ROOT}")
print(f"  Out dir    : {CFG.OUT_DIR}")
print(f"  SiT weights: {CFG.SIT_CACHE_PATH}")
print(f"  SiT size   : {Path(CFG.SIT_CACHE_PATH).stat().st_size/1e6:.1f} MB")


CFG loaded ✅
  Device     : cuda
  Dataset    : /user/HS402/mi00806/.cache/kagglehub/datasets/cjinny/mrnet-v1/versions/1/MRNet-v1.0
  Out dir    : /user/HS402/mi00806/Downloads/AML/mrnet_outputs
  SiT weights: /user/HS402/mi00806/Downloads/AML/sit_zip/SiT_Small_ImageNet.pth
  SiT size   : 669.7 MB


## Section 4 — Dataset Labels & Stratified Split

MRNet provides three CSV files (one per condition) for the train split.
Labels: ACL tear, Meniscus tear, Abnormal (multi-label — not mutually exclusive).
We perform a **stratified 85/15 split** on the `abnormal` label.


In [5]:
def load_labels(root, split):
    """Load and merge ACL / Abnormal / Meniscus CSVs into one DataFrame."""
    acl      = pd.read_csv(root / f"{split}-acl.csv",
                           header=None, names=["id", "acl"])
    abnormal = pd.read_csv(root / f"{split}-abnormal.csv",
                           header=None, names=["id", "abnormal"])
    meniscus = pd.read_csv(root / f"{split}-meniscus.csv",
                           header=None, names=["id", "meniscus"])
    df = acl.merge(abnormal, on="id").merge(meniscus, on="id")
    df["id"] = df["id"].astype(str).str.zfill(4)
    for plane in ["sagittal", "coronal", "axial"]:
        df[f"{plane}_path"] = df["id"].apply(
            lambda x, p=plane: str(root / "train" / p / f"{x}.npy"))
    df["target"] = df.apply(
        lambda r: [float(r["acl"]),
                   float(r["meniscus"]),
                   float(r["abnormal"])], axis=1)
    return df


all_df = load_labels(CFG.ROOT, "train")
train_df, valid_df = train_test_split(
    all_df, test_size=0.15, random_state=CFG.SEED,
    stratify=all_df["abnormal"].astype(int)
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(f"Train : {len(train_df)} scans")
print(f"Valid : {len(valid_df)} scans  (15% stratified split)")
print()
print("Label distribution:")
for lbl in CFG.LABEL_NAMES:
    tr_n = int(train_df[lbl].sum())
    va_n = int(valid_df[lbl].sum())
    print(f"  {lbl:>10}:  "
          f"train {tr_n:>3} pos ({100*tr_n/len(train_df):.1f}%)  "
          f"| valid {va_n:>3} pos ({100*va_n/len(valid_df):.1f}%)")


Train : 960 scans
Valid : 170 scans  (15% stratified split)

Label distribution:
         acl:  train 185 pos (19.3%)  | valid  23 pos (13.5%)
    meniscus:  train 328 pos (34.2%)  | valid  69 pos (40.6%)
    abnormal:  train 776 pos (80.8%)  | valid 137 pos (80.6%)


## Section 6 — Preprocessing & Dataset Class

**Slice selection:** Top-5 variance slices selected per volume → 3 chosen
evenly → stacked as pseudo-RGB image. High-variance slices contain structural
detail rather than blank background.

**Augmentation (train only):** rotation ±20°, translation ±10%,
scale 85–115%, horizontal flip, colour jitter.


In [8]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def select_topk_variance_slices(volume, top_k=5, n_out=3):
    D   = volume.shape[0]
    var = np.var(volume.reshape(D,-1), axis=1)
    topk = np.sort(np.argsort(var)[-min(top_k,D):])
    n_out = min(n_out, len(topk))
    chosen = topk[np.linspace(0, len(topk)-1, n_out, dtype=int)]
    if len(chosen) < n_out:
        chosen = np.pad(chosen, (0, n_out-len(chosen)), mode="edge")
    return volume[chosen]


def volume_to_rgb(volume, top_k=5, n_ch=3):
    slices = select_topk_variance_slices(volume, top_k=top_k, n_out=n_ch)
    channels = []
    for s in slices:
        s = s.astype(np.float32)
        lo, hi = s.min(), s.max()
        s = (s-lo)/(hi-lo+1e-8)
        channels.append((s*255).clip(0,255).astype(np.uint8))
    return Image.fromarray(np.stack(channels, axis=-1))


train_tfm = T.Compose([
    T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    T.RandomAffine(degrees=20, translate=(0.10,0.10), scale=(0.85,1.15), fill=0),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.25, contrast=0.25),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
valid_tfm = T.Compose([
    T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class MRNetDataset(Dataset):
    """Loads sag/cor/axi volumes and returns 3 pseudo-RGB tensors + target."""
    def __init__(self, df, transforms=None, top_k=5, n_ch=3):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.top_k = top_k; self.n_ch = n_ch

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        imgs = {}
        for plane in ["sagittal","coronal","axial"]:
            vol = np.load(row[f"{plane}_path"], mmap_mode="r")
            img = volume_to_rgb(vol, top_k=self.top_k, n_ch=self.n_ch)
            if self.transforms: img = self.transforms(img)
            imgs[plane] = img
        target = torch.tensor(row["target"], dtype=torch.float32)
        return imgs["sagittal"], imgs["coronal"], imgs["axial"], target, row["id"]


s, c, a, t, eid = MRNetDataset(train_df.head(2), transforms=train_tfm)[0]
print(f"Tensor shape : {tuple(s.shape)}")
print(f"Target       : {t.tolist()}")
print(f"Exam ID      : {eid}")
print("Preprocessing ready ✅")


Tensor shape : (3, 224, 224)
Target       : [0.0, 1.0, 1.0]
Exam ID      : 0104
Preprocessing ready ✅


## Section 7 — Class Imbalance: WeightedRandomSampler + DataLoaders

ACL tears appear in only ~18% of scans. `WeightedRandomSampler` assigns
each scan a weight inversely proportional to its rarest positive label
frequency, so ACL-positive scans appear ~4× more per epoch.


In [9]:
def compute_sample_weights(df, label_names):
    freqs = df[label_names].sum().values / len(df)
    weights = []
    for _, row in df.iterrows():
        labels = np.array([row[l] for l in label_names])
        pos_f  = freqs[labels == 1]
        w = 1.0/(pos_f.min() if len(pos_f)>0 else (1-freqs).mean())
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float64)


sample_weights = compute_sample_weights(train_df, CFG.LABEL_NAMES)
sampler = WeightedRandomSampler(weights=sample_weights,
                                num_samples=len(train_df), replacement=True)

label_counts = train_df[CFG.LABEL_NAMES].sum().values.astype(float)
neg_counts   = len(train_df) - label_counts
print("Neg/Pos ratio per label:")
for n, v in zip(CFG.LABEL_NAMES, neg_counts/label_counts):
    print(f"  {n:>10}: {v:.2f}x")

train_ds = MRNetDataset(train_df, transforms=train_tfm)
valid_ds = MRNetDataset(valid_df, transforms=valid_tfm)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(f"\nTrain : {len(train_ds)} scans → {len(train_loader)} batches/epoch")
print(f"Valid : {len(valid_ds)} scans → {len(valid_loader)} batches/epoch")
s, c, a, t, _ = next(iter(train_loader))
print(f"Batch shape per plane: {tuple(s.shape)}")
print("DataLoaders ready ✅")


Neg/Pos ratio per label:
         acl: 4.19x
    meniscus: 1.93x
    abnormal: 0.24x

Train : 960 scans → 120 batches/epoch
Valid : 170 scans → 22 batches/epoch
Batch shape per plane: (8, 3, 224, 224)
DataLoaders ready ✅
